In [ ]:
#All packages needed to run TwINFER simulation and inference are listed here. 
#If any of them are not installed, please install them using pip or conda env.
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import numba
import tqdm
import scipy
import seaborn
import scipy
import os
import sys
import joblib
from itertools import product
import joblib 
import importlib
import json

# Code to simulate a synthetic GRN and infer the network using TwINFER


## Details about the simulation

### Set this for both running the simulation and before inferring using TwINFER


In [ ]:
import twinfer
from twinfer.utils.paths import get_repo_root

path_to_code_repo = get_repo_root()  # <repo_root>/code/TwINFER, since __file__ -> package/twinfer/__init__.py, parents[2]


In [ ]:
#Path to TwINFER code repository
import os, sys

#Common path to data files
path_to_data = f"{path_to_code_repo}/simulation_example_input_data"
path_to_output = f"{path_to_code_repo}/simulation_example_output_data"

base_config = {
    'n_cells': 6000, #Number of cells before division (number of twin pairs)
    'simulation_time_before_division': 1500, #The time used to run the initial cells before division. User must set this time to ensure the population reaches steady state [hours]
    'twin_simulation_time_after_division': 48, #The time twin cells are simulated after division and measurements are stored in the output[hours]
    'twin_measurement_resolution': 1, #The time between each measurement of twin cells [hours]. For example, if twin_sampling_duration is 12 and twin_measurement_resolution is 1, the final dataframe will contain hourly measurements for 12 hours (0 is birth).
    "path_to_connectivity_matrix": f"{path_to_data}/connectivity_matrix_A_to_B.txt", #path to the connectivity matrix specifying the GRN to simulate
    "param_csv": f"{path_to_data}/median_parameter.csv", #Path to the parameters for all genes and interaction terms
    "rows_to_use": [[0,0]], #Rows in the parameter's csv file for each gene. Example - [0,0] will mean use row 0 parameters for both gene 1 and 2. The length should be equal to number of genes in the system. Ensure that each row in the parameter.csv has unique index.
    "output_folder": f"{path_to_output}", #Path to the output folder
    "log_file": f"{path_to_output}/log.jsonl", #Path to the log file
    "multiple_interaction_type": "additive", #Model if there are multiple regulators acting on a single gene - options are additive, or, and.
    "type": "A_to_B",  # Name of the network used -- will be in the filename
    "number_of_parallel_parameters": 1, #Number of parameters to be run in parallel
    "number_of_cores_per_parameter": 2, #Number of cores to be used per parameter (number_of_parallel_parameters * number_of_cores_per_parameter = number of cores in your computer)
}

## Functions and packages used in this notebook - run this everytime notebook is restarted


### For the simulation


In [ ]:
from joblib import Parallel, delayed
from tqdm.auto import tqdm
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numba import set_num_threads, get_num_threads

set_num_threads(base_config['number_of_cores_per_parameter'])
print("Threads Numba will use:", get_num_threads())

import importlib
from twinfer.simulation import gillespie_simulations
importlib.reload(gillespie_simulations)
from twinfer.simulation.gillespie_simulations import process_param_set

### For inferring with TwINFER


In [ ]:
# Calculation functions
import importlib
from twinfer.inference import correlation_functions
from twinfer.inference import infer

importlib.reload(correlation_functions)
importlib.reload(infer)

from twinfer.inference.correlation_functions import (
    calculate_pairwise_gene_gene_correlation_matrix,
    check_system_in_steady_state,
    check_gene_gene_correlation_threshold,
    calculate_twin_random_pair_correlations,
    differentiate_single_state_reg_and_multiple_states,
    identify_reg_if_multiple_states,
    get_cross_correlations,
    identify_actual_directed_edges,
    extract_param_index,
    read_input_matrix,
    plot_matrix_as_heatmap,
    print_summary,
)

from twinfer.inference.infer import (
    infer_with_twinfer
)

import json
from twinfer.utils.json_utils import make_json_safe

## Simulate the gene expression in a population of cells

The code simulates gene expression based on a GRN (described by the interaction matrix) and expression of each gene is defined by parameters (each row in the parameter sheet) using the Gillespie algorithm.


In [ ]:
os.makedirs(base_config['output_folder'], exist_ok=True)
rows_to_use = base_config['rows_to_use']
labels = ["rows_" + "_".join(map(str, row)) for row in rows_to_use]
path_to_simulation_file = process_param_set(rows_to_use[0], labels[0], base_config)
print(f"Saved the simulation file as {path_to_simulation_file}")

# Using TwINFER to infer network from the simulation

### Before starting, set the time-points used as t1 and t2. The default is 1 hour and 20 hours after division.

TwINFER uses three kinds of correlations used for distinguishing multiple-states and regulation.

- First, calculate gene-gene correlations $\rho$ for all pairs of genes at time t1. This will identify gene-pairs that are potentially interacting.
- Second, calculate twin pair correlation $\hat{\rho}_{\Delta}$ and random pair correlation. Comparing these two for all gene pairs will detect the existence of multiple states in the population vs just regulation.
- Finally, if there are multiple states in the population, compare twin pair correlation $\hat{\rho}_{\Delta}$ at time t1 and time t2. This will detect the existence of regulatory interaction.

Next, if there is regulation and single-state in the population, TwINFER can identify direction of regulatory interaction. This is done be comparing cross-correlations between twins at the two timepoints.

### Based on this algorithm, a network will be inferred and plotted.


### Input to TwINFER


### Run TwINFER for inference using the simulation, base_config, t1 and t2.


In [ ]:
from twinfer.utils.paths import get_data_root as _twinfer_get_data_root
TWINFER_PROJECT_ROOT = _twinfer_get_data_root().parent  # [2026-09-30 added: replaces hardcoded project-root paths (/home/gzu5140/TwINFER_KA, /gpfs/projects/b1255/hzhang/TwINFER_KA, old Keerthana_b1042 tree)]
path_to_simulation_file = f"{TWINFER_PROJECT_ROOT}/clean_data/simulation_example_output_data/df_rows_0_0_20112025_222419_ncells_6000_A_to_B_4e702629.csv"
twinfer_kwargs = {
    "path_to_simulation_file": path_to_simulation_file,
    "base_config": base_config,
    "t1": 1,  #time [hours] after division when t1 sample is collected
    "t2": 20, #time [hours] after division when t2 sample is collected
    "check_for_steady_state": True,
    "steady_state_tol": 0.01,
    "steady_state_flat_tol": None,
    "steady_state_min_r2": 0.80,
    "threshold_gene_gene_corr": 0.04, #Use direct threshold (used ONLY if use_scramble is False)
    "use_scramble": True, #If set to true, set the p_val_threshold_scrambled_gene_correlation;threshold_gene_gene_corr will NOT be used
    "p_val_threshold_scrambled_gene_correlation": 0.02, #used ONLY if use_scramble is True
    "show_scrambled_distribution_gene_correlation": True, 
    "z_score_threshold_two_states": 10, #Z-score threshold to detect multi-states in the system
    "p_value_threshold_cross_correlation": 0.01,
    "plot_correlation_matrices_as_heatmap": True,
    "have_any_output": True,
    "seed": 101010,
    "infer_direction_for_which_edges": "all-edges", #can be either single-state, all-potential-regulation (gene correlation is significant) or all-edges,
    "merge_time_points": True, #If True, then cells from the two timepoints will be used to calculate gene correlations and random-pair difference correlations
    "n_cores": 10,
}

In [ ]:
correlation_matrices = infer_with_twinfer(**twinfer_kwargs)

## Export the matrices as a json file for reuse

In [ ]:
json_safe = {
    k: make_json_safe(v)
    for k, v in correlation_matrices.items()
}

path_to_json_file = f"{path_to_output}/{base_config['type']}_correlation_matrices.json"
with open(path_to_json_file, "w") as f:
    json.dump(json_safe, f, indent=2)

## Plotting and saving correlation matrices

In [ ]:
#Set path to plots if you want to save them - if None, the plots will be displayed, not saved
path_to_plots = None

In [ ]:
# path_to_json_file = f"{path_to_output}/{base_config['type']}_correlation_matrices.json"
with open(path_to_json_file, "r") as f:
    correlation_matrices = json.load(f)

print(correlation_matrices.keys())


### Helper functions and settings for the plot

In [ ]:
import matplotlib.font_manager as fm
from pathlib import Path
import twinfer.plotting as twinfer_plotting
# ============================================================
# Fonts / style
# ============================================================
fonts_dir = Path(twinfer_plotting.__file__).parent / "assets" / "fonts"
font_paths = sorted(fonts_dir.glob("*.ttf"))

for fp in font_paths:
    try:
        fm.fontManager.addfont(str(fp))
        print("✔ Loaded font:", fp)
    except Exception as e:
        print("⚠️  Could not load:", fp, "|", e)

# ==== LaTeX + SVG text mode (Illustrator-safe) ====
# ==== LaTeX + SVG text mode (Illustrator-safe) ====
import matplotlib.pyplot as plt
plt.rcParams['pdf.fonttype'] = 42  # For PDF export
plt.rcParams['ps.fonttype'] = 42   # For PostScript (EPS) export
plt.rcParams['font.sans-serif'] = ["Arial"]
plt.rcParams['font.family'] = "sans-serif"
plt.rcParams['svg.fonttype'] = "none"
plt.rcParams['mathtext.fontset'] = "cm"
plt.rcParams['axes.labelsize'] = 18*1.5      # x/y labels
plt.rcParams['axes.titlesize'] = 20*1.5 
plt.rcParams['xtick.labelsize'] = 12*1.5     # x-axis tick labels
plt.rcParams['ytick.labelsize'] = 12*1.5    # y-axis tick labels
plt.rcParams['legend.fontsize'] = 12*1.5     # legend text

In [ ]:
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap, ListedColormap
from matplotlib.patches import Rectangle
import seaborn as sns
from pathlib import Path

from twinfer.plotting import network_plots
importlib.reload(network_plots)
from twinfer.plotting.network_plots import make_reds_blues_colormap

In [ ]:
pairwise_gene_gene_correlation_matrix = pd.DataFrame(correlation_matrices['pairwise_gene_gene_correlation_matrix'])
all_gene_pairs = correlation_matrices['all_gene_pairs']
final_directed_edges = correlation_matrices['final_directed_edges']
potential_regulation = correlation_matrices['potential_regulation']
filtered_direction_matrix = pd.DataFrame(correlation_matrices['direction_matrix'])
multiple_states_and_reg = correlation_matrices['gene_lists']['multiple_states_and_reg']

In [ ]:
#Plot gene-correlation data
gene_list = sorted(
    {g for pair in all_gene_pairs for g in pair},
    key=lambda x: int(x.split("_")[1])
)
gene_labels = [f"g{g.split('_')[1]}" for g in gene_list]
pairwise_gene_gene_correlation_matrix = pairwise_gene_gene_correlation_matrix.loc[gene_list, gene_list]
data_matrix = pairwise_gene_gene_correlation_matrix.to_numpy(float)
np.fill_diagonal(data_matrix, np.nan)
fig = plt.figure(figsize=(9,8))
gs = fig.add_gridspec(2, 1, height_ratios=[0.05, 0.95], hspace=0.1)
cbar_ax = fig.add_subplot(gs[0])
heatmap_ax = fig.add_subplot(gs[1])

plot_matrix = data_matrix.copy()
plot_matrix[:] = 0.0

# Restore only potential regulation correlations
for g1, g2 in potential_regulation:
    if g1 in gene_list and g2 in gene_list:
        i = gene_list.index(g1)
        j = gene_list.index(g2)
        plot_matrix[i, j] = data_matrix[i, j]
        plot_matrix[j, i] = data_matrix[j, i]
np.fill_diagonal(plot_matrix, np.nan)
vmin = -0.299
vmax = 0.299

cmap = make_reds_blues_colormap(vmin = vmin, vmax = vmax)
cmap.set_bad(color="#D9D9D9")

gene_to_label = {
    "gene_1": r"$A$",
    "gene_2": r"$B$",
    "gene_3": r"$C$",
    "gene_4": r"$D$",
    "gene_5": r"$E$",
}

plot_df = pd.DataFrame(
    plot_matrix,
    index=gene_list,
    columns=gene_list
)

sns.heatmap(
    plot_df,
    ax=heatmap_ax,
    cmap=cmap,
    vmin=vmin,
    vmax=vmax,
    square=True,
    cbar=True,
    cbar_ax=cbar_ax,
    cbar_kws={'orientation': 'horizontal'},
    linewidths=0.5,
    linecolor="black",
    # center=center
)

gene_to_idx = {g: i for i, g in enumerate(gene_list)}
for g1, g2 in potential_regulation:
    if g1 in gene_to_idx and g2 in gene_to_idx:
        i = gene_to_idx[g1]
        j = gene_to_idx[g2]

        for (x, y) in [(j, i), (i, j)]:
            rect = Rectangle(
                (x, y), 1, 1,
                fill=False,
                edgecolor="black",
                linewidth=1.5,
                clip_on=False
            )
            heatmap_ax.add_patch(rect)
        
cbar_ax.xaxis.set_label_position('top')
cbar_ax.set_xlabel(r"gene correlation $\rho$", labelpad=10)
cbar_ax.xaxis.tick_top()
xlabels = [gene_to_label.get(g, g) for g in plot_df.columns]
ylabels = [gene_to_label.get(g, g) for g in plot_df.index]

heatmap_ax.set_xticklabels(xlabels, rotation=90, ha="center", va="top")
heatmap_ax.set_yticklabels(ylabels, rotation=0, ha="right", va="center")
heatmap_ax.set_aspect("equal")
# --- transparent background ---
# plt.tight_layout()
fig.patch.set_alpha(0)
for ax in [heatmap_ax, cbar_ax]:
    ax.set_facecolor("none")
for im in heatmap_ax.get_images() + cbar_ax.get_images():
    im.set_facecolor((1, 1, 1, 0))
    im.set_edgecolor((1, 1, 1, 0))
for spine in heatmap_ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1)
    spine.set_edgecolor('black')
    spine.set_clip_on(False)

# --- save ---
# [2026-10-01 commented out: stale path in the old Keerthana_b1042 tree; it overrode the default path_to_plots = None (set earlier in the notebook), so the tutorial would have tried to write plots there. Set path_to_plots to a folder to save plots.]
# path_to_plots= "/home/gzu5140/Keerthana_b1042/grnInference/plots/"
if path_to_plots:
    out = Path(path_to_plots)
    out.mkdir(parents=True, exist_ok=True)
    matrix_type = "gene_correlation"
    png_path = out / f"{matrix_type}.png"
    svg_path = out / f"{matrix_type}.svg"
    pdf_path = out / f"{matrix_type}.pdf"
    plt.savefig(png_path, dpi=300, bbox_inches='tight', transparent=True)
    fig.savefig(svg_path, bbox_inches='tight', transparent=True,
                facecolor='none', edgecolor='none', format='svg')
    fig.savefig(pdf_path, bbox_inches='tight', transparent=True,
                facecolor='none', edgecolor='none', format='pdf')
    print(f"Saved PNG: {png_path}")
    print(f"Saved SVG: {svg_path}")
    plt.show()
# # else:
plt.show()

In [ ]:
filtered_direction_matrix

In [ ]:
#Plot TwINFER output
gene_list = sorted(
    {g for pair in all_gene_pairs for g in pair},
    key=lambda x: int(x.split("_")[1])
)
gene_labels = [f"g{g.split('_')[1]}" for g in gene_list]
# direction_matrix = filtered_direction_matrix.loc[gene_list, gene_list]
data_matrix = filtered_direction_matrix.to_numpy(float)
masked_matrix = np.fill_diagonal(data_matrix, np.nan)
fig = plt.figure(figsize=(6,6))
gs = fig.add_gridspec(2, 1, height_ratios=[0.05, 0.95], hspace=0.1)
cbar_ax = fig.add_subplot(gs[0])
heatmap_ax = fig.add_subplot(gs[1])
plot_matrix = data_matrix.copy()
plot_matrix[:] = 0.0

# Restore only final-ege correlations
for g1, g2 in final_directed_edges:
    if g1 in gene_list and g2 in gene_list:
        i = gene_list.index(g1)
        j = gene_list.index(g2)
        plot_matrix[i, j] = data_matrix[i, j]
        plot_matrix[j, i] = data_matrix[j, i]
np.fill_diagonal(plot_matrix, np.nan)
vmin = -0.299
vmax = 0.299
cmap = make_reds_blues_colormap(vmin = vmin, vmax = vmax)

cmap.set_bad(color="#D9D9D9")
# --- draw heatmap ---
sns.heatmap(
    plot_matrix,
    ax=heatmap_ax,
    cmap=cmap,
    vmin=vmin,
    vmax=vmax,
    square=True,
    cbar=True,
    cbar_ax=cbar_ax,
    cbar_kws={'orientation': 'horizontal'},
    linewidths=0.5,
    linecolor="black",
    # center=center
)
for g1, g2 in multiple_states_and_reg:
    if g1 in gene_list and g2 in gene_list:
        i = gene_list.index(g1)
        j = gene_list.index(g2)
        print(i, j)

        # Draw diagonal in cell (i, j) - top-left to bottom-right
        heatmap_ax.plot(
            [j, j+1],      # x: left → right
            [i, i+1],      # y: top → bottom
            linestyle="--",
            color="black",
            linewidth=1.5,
            clip_on=False
        )
        
        # Draw diagonal in symmetric cell (j, i) - top-left to bottom-right
        heatmap_ax.plot(
            [i, i+1],      # x: left → right
            [j, j+1],      # y: top → bottom
            linestyle="--",
            color="black",
            linewidth=1.5,
            clip_on=False
        )

# --- labels ---
cbar_ax.xaxis.set_label_position('top')
cbar_ax.set_xlabel(r"$\hat{\rho}^{\dagger}_{x(d_{2}) \to y(d_{4})}$", labelpad=10)
cbar_ax.xaxis.tick_top()

heatmap_ax.set_xticklabels(gene_labels, rotation=90, ha="center", va="top")
heatmap_ax.set_yticklabels(gene_labels, rotation=0, ha="right", va="center")
# --- transparent background ---
# plt.tight_layout()
fig.patch.set_alpha(0)
for ax in [heatmap_ax, cbar_ax]:
    ax.set_facecolor("none")
for im in heatmap_ax.get_images() + cbar_ax.get_images():
    im.set_facecolor((1, 1, 1, 0))
    im.set_edgecolor((1, 1, 1, 0))
for spine in heatmap_ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1)
    spine.set_edgecolor('black')
    spine.set_clip_on(False)

# --- save ---
if path_to_plots:
    out = Path(path_to_plots)
    out.mkdir(parents=True, exist_ok=True)
    matrix_type = "direction_correlation"
    png_path = out / f"{matrix_type}.png"
    svg_path = out / f"{matrix_type}.svg"
    pdf_path = out / f"{matrix_type}.pdf"
    plt.savefig(png_path, dpi=300, bbox_inches='tight', transparent=True)
    fig.savefig(svg_path, bbox_inches='tight', transparent=True,
                facecolor='none', edgecolor='none', format='svg')
    fig.savefig(pdf_path, bbox_inches='tight', transparent=True,
                facecolor='none', edgecolor='none', format='pdf')
    print(f"Saved PNG: {png_path}")
    print(f"Saved SVG: {svg_path}")
    plt.show()
# # else:
plt.show()

In [ ]:
# ============================================================
# Visualize a TwINFER-inferred gene regulatory network
# ============================================================
#
# INPUTS
# ------
# 1. INFERRED_RESULT_PATH
#    Path to the JSON file exported by TwINFER:
#
#        <output_directory>/<network_name>_correlation_matrices.json
#
#    This JSON must contain:
#        - "filtered_direction_matrix"
#        - "final_directed_edges"
#
# 2. GROUND_TRUTH_PATH
#    Optional path to the ground-truth adjacency matrix.
#    The file may be whitespace-delimited (.txt) or comma-delimited (.csv).
#
#    Set this to None to plot only the inferred network.
#
# 3. OUTPUT_PATH
#    Optional path for saving the figure.
#    Supported examples:
#        "network_comparison.pdf"
#        "network_comparison.svg"
#        "network_comparison.png"
#
#    Set this to None to display the figure without saving it.
#
#
# OUTPUTS
# -------
# - Displays the inferred network.
# - If GROUND_TRUTH_PATH is provided, displays ground truth and
#   inference side by side using the same node layout.
# - Optionally saves the figure to OUTPUT_PATH.
# - Creates the variables:
#       inferred_matrix
#       ground_truth_matrix   (when ground truth is provided)
#       gene_list
#       node_labels
#       fig
#
# Matrix convention:
#       matrix[i, j] = gene i regulates gene j
#
#       positive value -> activation
#       negative value -> repression
#
# Self-regulation is removed from the visualization.
# ============================================================


# ------------------------------------------------------------
# User inputs
# ------------------------------------------------------------

# JSON file exported by the TwINFER inference notebook.
INFERRED_RESULT_PATH = (
    "/path/to/network_correlation_matrices.json"
)

# Optional ground-truth connectivity matrix.
# Set to None to plot only the inferred network.
GROUND_TRUTH_PATH = (
    "/path/to/ground_truth_connectivity_matrix.txt"
)

# Optional figure output path.
# Set to None if the figure should only be displayed.
OUTPUT_PATH = None

# Example:
# OUTPUT_PATH = "ground_truth_vs_inferred_network.pdf"


# ------------------------------------------------------------
# Imports
# ------------------------------------------------------------

from pathlib import Path
import json
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from twinfer.plotting.network_plots import plot_grn


# ------------------------------------------------------------
# Convert user inputs to Path objects
# ------------------------------------------------------------

INFERRED_RESULT_PATH = Path(INFERRED_RESULT_PATH)

if GROUND_TRUTH_PATH is not None:
    GROUND_TRUTH_PATH = Path(GROUND_TRUTH_PATH)

if OUTPUT_PATH is not None:
    OUTPUT_PATH = Path(OUTPUT_PATH)


# ------------------------------------------------------------
# Check that required input files exist
# ------------------------------------------------------------

if not INFERRED_RESULT_PATH.is_file():
    raise FileNotFoundError(
        "TwINFER result JSON was not found:\n"
        f"{INFERRED_RESULT_PATH}"
    )

if (
    GROUND_TRUTH_PATH is not None
    and not GROUND_TRUTH_PATH.is_file()
):
    raise FileNotFoundError(
        "Ground-truth matrix was not found:\n"
        f"{GROUND_TRUTH_PATH}"
    )


# ------------------------------------------------------------
# Load the exported TwINFER result
# ------------------------------------------------------------

with INFERRED_RESULT_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    correlation_matrices = json.load(file)


# Check that the JSON contains the information needed to
# reconstruct the final inferred network.

required_keys = {
    "filtered_direction_matrix",
    "final_directed_edges",
}

missing_keys = required_keys.difference(
    correlation_matrices.keys()
)

if missing_keys:
    raise KeyError(
        "The TwINFER result JSON is missing the following "
        f"required keys: {sorted(missing_keys)}"
    )


# ------------------------------------------------------------
# Reconstruct the unfiltered signed direction matrix
# ------------------------------------------------------------
# When a pandas DataFrame is exported to JSON through to_dict(),
# it becomes a nested dictionary. pd.DataFrame reconstructs it
# with the original row and column labels.

direction_matrix = pd.DataFrame(
    correlation_matrices[
        "filtered_direction_matrix"
    ]
).astype(float)

direction_matrix.index = (
    direction_matrix.index.map(str)
)

direction_matrix.columns = (
    direction_matrix.columns.map(str)
)


# ------------------------------------------------------------
# Determine a consistent gene order
# ------------------------------------------------------------
# Natural sorting places:
#
#     gene_2 before gene_10
#
# rather than using ordinary alphabetical sorting.

def natural_sort_key(label):
    """Return a key for natural alphanumeric sorting."""

    components = re.split(
        r"(\d+)",
        str(label),
    )

    return [
        int(component)
        if component.isdigit()
        else component.lower()
        for component in components
    ]


gene_list = sorted(
    set(direction_matrix.index)
    | set(direction_matrix.columns),
    key=natural_sort_key,
)

direction_matrix = direction_matrix.reindex(
    index=gene_list,
    columns=gene_list,
    fill_value=0.0,
)

gene_to_index = {
    gene: index
    for index, gene in enumerate(gene_list)
}


# ------------------------------------------------------------
# Construct the final inferred adjacency matrix
# ------------------------------------------------------------
# `filtered_direction_matrix` contains signed scores for all
# tested gene pairs.
#
# `final_directed_edges` contains only the edges retained by
# TwINFER.
#
# Therefore, this section copies scores only for retained edges
# and leaves all other entries equal to zero.

inferred_matrix = np.zeros(
    (
        len(gene_list),
        len(gene_list),
    ),
    dtype=float,
)

for edge in correlation_matrices[
    "final_directed_edges"
]:
    if len(edge) != 2:
        raise ValueError(
            "Every item in `final_directed_edges` must "
            f"contain exactly two genes. Invalid edge: {edge}"
        )

    source_gene, target_gene = map(
        str,
        edge,
    )

    if source_gene not in gene_to_index:
        raise KeyError(
            "An inferred edge contains an unknown source "
            f"gene: {source_gene}"
        )

    if target_gene not in gene_to_index:
        raise KeyError(
            "An inferred edge contains an unknown target "
            f"gene: {target_gene}"
        )

    # Self-regulation is not counted or displayed.
    if source_gene == target_gene:
        continue

    source_index = gene_to_index[
        source_gene
    ]

    target_index = gene_to_index[
        target_gene
    ]

    inferred_matrix[
        source_index,
        target_index,
    ] = direction_matrix.loc[
        source_gene,
        target_gene,
    ]

np.fill_diagonal(
    inferred_matrix,
    0.0,
)


# ------------------------------------------------------------
# Create shorter display labels
# ------------------------------------------------------------
# gene_1, gene_2, ... are displayed as g1, g2, ...
# Other gene names are left unchanged.

node_labels = []

for gene in gene_list:
    gene_number = re.fullmatch(
        r"gene_(\d+)",
        gene,
    )

    if gene_number:
        node_labels.append(
            f"g{gene_number.group(1)}"
        )
    else:
        node_labels.append(gene)


# ------------------------------------------------------------
# Create one fixed node layout
# ------------------------------------------------------------
# The same positions are used in both panels so that ground
# truth and inferred networks can be compared directly.

angles = np.linspace(
    np.pi / 2,
    np.pi / 2 - 2 * np.pi,
    len(gene_list),
    endpoint=False,
)

fixed_positions = {
    index: np.array(
        [
            0.5 + 0.40 * np.cos(angle),
            0.5 + 0.40 * np.sin(angle),
        ]
    )
    for index, angle in enumerate(angles)
}


# ------------------------------------------------------------
# Helper for loading the ground-truth matrix
# ------------------------------------------------------------

def load_ground_truth_matrix(
    matrix_path,
    expected_shape,
):
    """
    Load a square numeric matrix from a whitespace-delimited
    or comma-delimited text file.
    """

    # First try ordinary whitespace-delimited text.
    try:
        matrix = np.loadtxt(
            matrix_path,
            dtype=float,
        )

    # If that fails, try a comma-delimited file.
    except ValueError:
        matrix = np.loadtxt(
            matrix_path,
            delimiter=",",
            dtype=float,
        )

    matrix = np.asarray(
        matrix,
        dtype=float,
    )

    if matrix.ndim != 2:
        raise ValueError(
            "The ground-truth input must be a "
            f"two-dimensional matrix; received shape {matrix.shape}."
        )

    if matrix.shape[0] != matrix.shape[1]:
        raise ValueError(
            "The ground-truth matrix must be square; "
            f"received shape {matrix.shape}."
        )

    if matrix.shape != expected_shape:
        raise ValueError(
            "The ground-truth and inferred matrices must "
            "have the same shape:\n"
            f"    ground truth: {matrix.shape}\n"
            f"    inferred:     {expected_shape}"
        )

    if not np.all(np.isfinite(matrix)):
        raise ValueError(
            "The ground-truth matrix contains NaN or "
            "infinite values."
        )

    # Self-regulation is not counted or displayed.
    np.fill_diagonal(
        matrix,
        0.0,
    )

    return matrix


# ------------------------------------------------------------
# Plot inferred network, with optional ground-truth comparison
# ------------------------------------------------------------

if GROUND_TRUTH_PATH is None:
    # Inferred network only.

    ground_truth_matrix = None

    fig, ax = plt.subplots(
        figsize=(7, 6),
    )

    plot_grn(
        inferred_matrix,
        node_labels=node_labels,
        title="Inferred network",
        ax=ax,
        fixed_pos=fixed_positions,
    )

else:
    # Load and validate the ground-truth matrix.

    ground_truth_matrix = (
        load_ground_truth_matrix(
            GROUND_TRUTH_PATH,
            expected_shape=inferred_matrix.shape,
        )
    )

    # Plot ground truth and inference side by side.

    fig, axes = plt.subplots(
        nrows=1,
        ncols=2,
        figsize=(14, 6),
    )

    plot_grn(
        ground_truth_matrix,
        node_labels=node_labels,
        title="Ground truth",
        ax=axes[0],
        fixed_pos=fixed_positions,
    )

    plot_grn(
        inferred_matrix,
        node_labels=node_labels,
        title="Inferred network",
        ax=axes[1],
        fixed_pos=fixed_positions,
    )


fig.tight_layout()


# ------------------------------------------------------------
# Optionally save the figure
# ------------------------------------------------------------

if OUTPUT_PATH is not None:
    OUTPUT_PATH.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    fig.savefig(
        OUTPUT_PATH,
        bbox_inches="tight",
        dpi=300,
    )

    print(
        f"Figure saved to:\n{OUTPUT_PATH.resolve()}"
    )


# ------------------------------------------------------------
# Report the loaded network
# ------------------------------------------------------------

print(
    f"Number of genes: {len(gene_list)}"
)

print(
    "Gene order:",
    gene_list,
)

print(
    "Number of inferred non-self edges:",
    int(np.count_nonzero(inferred_matrix)),
)

plt.show()